# 03 XGBoost: receiver risk model + reasons

Work card 03. Training runs as a background script in the **Jupyter terminal** (a few minutes on 3 CPUs):

```bash
cd ~/shared/trustpause
nohup python -m src.train_xgb > data/train_xgb.log 2>&1 &
tail -f data/train_xgb.log
```

Wait for `Saved .../xgb.json`, then run the cells below. Don't run other heavy jobs on the server meanwhile.

In [ ]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import precision_recall_curve

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.features import FEATURE_NAMES
from src.reasons import reasons_for

metrics = json.loads((ROOT / 'models' / 'metrics.json').read_text())
thresholds = json.loads((ROOT / 'models' / 'thresholds.json').read_text())
print(f"Best iteration {metrics['best_iteration']}, trained in {metrics['train_seconds']} s")
print(f"Single prediction latency: p50 {metrics['single_predict_ms_p50']} ms, p95 {metrics['single_predict_ms_p95']} ms")
print(f"Thresholds: pause >= {thresholds['pause']:.4f}, hold >= {thresholds['hold']:.4f}")

## Headline metrics

Fraud is ~0.1% of transfers, so PR-AUC and precision/recall matter, not accuracy. Thresholds were chosen on **validation**; test is reported untouched.

In [ ]:
rows = []
for split in ('val', 'test'):
    m = metrics[split]
    for name in ('pause_band', 'hold_band'):
        rows.append({'split': split, 'band': name.replace('_band', ''), 'pr_auc': m['pr_auc'], 'roc_auc': m['roc_auc'], **m[name]})
pd.DataFrame(rows)

## Precision / recall at different alert budgets

`alert_rate_%` = share of all transfers flagged. Use this to choose the pause and hold budgets (`--pause-rate`, `--hold-rate`).

In [ ]:
for split in ('val', 'test'):
    print(split)
    display(pd.DataFrame(metrics[split]['by_alert_rate']))

In [ ]:
import matplotlib.pyplot as plt

scores = pd.read_pickle(ROOT / 'data' / 'xgb_scores.pkl')
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for split in ('val', 'test'):
    s = scores[scores['split'] == split]
    p, r, _ = precision_recall_curve(s['label'], s['score_xgb'])
    axes[0].plot(r, p, label=f"{split} (PR-AUC {metrics[split]['pr_auc']})")
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision'); axes[0].set_title('XGBoost precision-recall'); axes[0].legend()

imp = pd.Series(metrics['importance_gain']).head(12)[::-1]
axes[1].barh(imp.index, imp.values, color='#0F6E78'); axes[1].set_title('Feature importance (gain)')
plt.tight_layout()
plt.savefig(ROOT / 'screenshots' / 'xgb_pr_and_importance.png', dpi=150)
plt.show()

## Reload check and example explanations

Reasons come from XGBoost's own `pred_contribs` (SHAP values), so no `shap` package is needed. Sentences contain counts and amounts only, never account IDs.

In [ ]:
booster = xgb.Booster()
booster.load_model(ROOT / 'models' / 'xgb.json')

features = pd.read_pickle(ROOT / 'data' / 'features_all.pkl')
test = features[features['split'] == 'test']
del features

saved = scores.loc[test.index[:1000], 'score_xgb'].to_numpy()
again = booster.inplace_predict(test[FEATURE_NAMES].iloc[:1000].to_numpy(dtype=np.float32))
print('Reloaded model gives the same scores:', np.allclose(saved, again, atol=1e-6))

In [ ]:
def decision(score):
    return 'HOLD' if score >= thresholds['hold'] else 'PAUSE' if score >= thresholds['pause'] else 'ALLOW'

test_scores = scores[scores['split'] == 'test']
examples = pd.concat([
    test_scores[test_scores['label'] == 1].nlargest(3, 'score_xgb'),
    test_scores[test_scores['label'] == 1].sample(1, random_state=0),
    test_scores[test_scores['label'] == 0].nlargest(1, 'score_xgb'),
])
explained = reasons_for(booster, test.loc[examples.index, FEATURE_NAMES], FEATURE_NAMES)
for (idx, row), reasons in zip(examples.iterrows(), explained):
    kind = 'laundering' if row['label'] else 'normal'
    print(f"score {row['score_xgb']:.4f} -> {decision(row['score_xgb'])}  (actually {kind})")
    for r in reasons:
        print(f"   [{r['code']}] {r['sentence']}")
    print()